In [10]:
%pip install openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [1]:
import pandas as pd

df = pd.read_excel(r"C:\Users\Owner\Downloads\multidialectal_travel_corpus.xlsx")

print(df.head())
print("عدد الصفوف:", len(df))


                                    Column1  \
0                    Modern Standard Arabic   
1    سأحجز رحلتي عبر الإنترنت لتوفير الوقت.   
2        أبحث عن فندق بخدمة واي فاي مجانية.   
3               هل يمكنني تغيير موعد رحلتي؟   
4  أريد استكشاف الثقافة المحلية لهذا البلد.   

                                             Column2  \
0                                      Saudi Dialect   
1  - Saudi: راح أحجز رحلتي عن طريق النت عشان أوفر...   
2    - Saudi: أدوّر عن فندق فيه خدمة واي فاي مجانية.   
3                  - Saudi: هل أقدر أغير موعد رحلتي؟   
4                - Saudi: أبغى أكتشف ثقافة البلد ذي.   

                                             Column3  \
0                                   Egyptian Dialect   
1  - Egyptian: هحجز رحلتي عن طريق النت عشان أوفر ...   
2  - Egyptian: بدوّر على فندق فيه خدمة واي فاي مج...   
3                  - Egyptian: ينفع أغير معاد رحلتي؟   
4         - Egyptian: أنا عايز أكتشف ثقافة البلد دي.   

                                           

In [2]:
import pandas as pd
import re


In [3]:
df = pd.read_excel(r"C:\Users\Owner\Downloads\multidialectal_travel_corpus.xlsx")


In [5]:
df = df.drop(index=0).reset_index(drop=True)


In [6]:
df.columns = [
    "msa",
    "saudi",
    "egyptian",
    "iraqi",
    "levantine",
    "moroccan"
]


In [7]:
def remove_dialect_tag(text):
    if pd.isna(text):
        return text
    return re.sub(r"-\s*\w+:\s*", "", text).strip()

dialect_cols = ["saudi", "egyptian", "iraqi", "levantine", "moroccan"]

for col in dialect_cols:
    df[col] = df[col].apply(remove_dialect_tag)


In [8]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

for col in df.columns:
    df[col] = df[col].astype(str).apply(normalize_spaces)


In [9]:
df = df.dropna()


In [10]:
df = df[df["msa"].str.len() > 10]


In [11]:
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

for col in df.columns:
    df[col] = df[col].apply(normalize_arabic)


In [12]:
print("عدد الصفوف بعد التنظيف:", len(df))
print(df.sample(3))


عدد الصفوف بعد التنظيف: 49885
                                     msa                            saudi  \
32153      اريد ان استاجر سياره مع سايق.       ابغي استاجر سياره مع سواق.   
20777  ما هي الاماكن التي تنصح بزيارتها؟  وش الاماكن اللي تنصح بزياراتها؟   
21826    هل يوجد متجر لبيع الالكترونيات؟         فيه محل يبيع الكترونيات؟   

                                egyptian                             iraqi  \
32153           عايز ااجر عربيه مع سواق.        اريد ااجر سياره ويّا سايق.   
20777  ايه الاماكن اللي بتنصح بزياراتها؟  شنو الاماكن اللي تنصح بزياراتها؟   
21826          فيه محل بيبيع الكترونيات؟          اكو محل يبيع الكترونيات؟   

                            levantine                              moroccan  
32153       بدي استاجر سياره مع سواق.           بغيت نكري طونوبيل مع شيفور.  
20777  شو الاماكن اللي بتنصح بزيارهن؟     شنو البلايص اللي كتْنصح بزيارتهم؟  
21826        في محل بيبيع الكترونيات؟  واش كاين شي محل كايبيع الالكترونيات؟  


In [13]:
df.to_csv(
    r"C:\Users\Owner\Desktop\NLP\clean_multidialectal_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ تم حفظ الملف بنجاح")


✅ تم حفظ الملف بنجاح


In [14]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset


Dataset({
    features: ['msa', 'saudi', 'egyptian', 'iraqi', 'levantine', 'moroccan', '__index_level_0__'],
    num_rows: 49885
})

In [15]:
dataset.save_to_disk("multidialectal_dataset")


Saving the dataset (0/1 shards):   0%|          | 0/49885 [00:00<?, ? examples/s]

In [16]:
translation_pairs = df[["msa", "saudi"]].rename(
    columns={"msa": "source", "saudi": "target"}
)


In [17]:
translation_pairs.sample(5)


,source,target
16262,هل يوجد ثلوج في الشتاء؟,فيه ثلج بالشتا؟
27814,اريد شراء بعض الاحذيه.,ابغي اشتري كم حذاء.
11482,سازور الاماكن التاريخيه والاثريه.,بزور الاماكن التاريخيه والاثريه.
27830,هل يوجد سينما قريبه؟,فيه سينما قريبه؟
40023,اريد زياره الاماكن الهاديه.,ابغي ازور الاماكن الهاديه.


In [18]:
rows = []

for _, row in df.iterrows():
    for dialect in ["saudi", "egyptian", "iraqi", "levantine", "moroccan"]:
        rows.append({
            "text": row[dialect],
            "label": dialect
        })

dialect_df = pd.DataFrame(rows)
dialect_df.sample(5)


,text,label
24183,رح نزل صور سفري علي السوشيال ميديا.,levantine
530,ابغي اعيش تجارب ما تُنسي.,saudi
55062,ادوّر علي رحلات رخيصه الي لندن.,iraqi
88067,شنو النشاطات الثقافيه المتوفره؟,iraqi
178234,بغيت نزور الغابات.,moroccan


In [19]:
label_map = {
    "saudi": "SA",
    "egyptian": "EG",
    "iraqi": "IQ",
    "levantine": "LEV",
    "moroccan": "MA"
}

dialect_df["label"] = dialect_df["label"].map(label_map)
dialect_df.sample(5)


,text,label
26941,هاروح اصطاد سمك.,EG
50457,اكو مواصلات عامه؟,IQ
184507,راح احجز تكسي حتي اروح للمتحف.,IQ
82017,راح احجز رحله بحريه.,IQ
35020,بسافر بالباص للمحطه الجايه.,SA


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dialect_df["label_id"] = le.fit_transform(dialect_df["label"])

dialect_df.head()


,text,label,label_id
0,راح احجز رحلتي عن طريق النت عشان اوفر وقت.,SA,4
1,هحجز رحلتي عن طريق النت عشان اوفر وقت.,EG,0
2,راح احجز سفري عن طريق النت حتي اوفر وقت.,IQ,1
3,رح احجز سفري عن طريق النت مشان وفر وقت.,LEV,2
4,غادي نحجز سفري عن طريق الانترنت باش نوفر الوقت.,MA,3


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dialect_df["text"],
    dialect_df["label_id"],
    test_size=0.2,
    random_state=42,
    stratify=dialect_df["label_id"]
)


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [23]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1,2),
        max_features=50000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        n_jobs=-1
    ))
])


In [24]:
model.fit(X_train, y_train)


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [25]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


Accuracy: 0.9011927433096121
              precision    recall  f1-score   support

          EG       0.89      0.89      0.89      9977
          IQ       0.90      0.90      0.90      9977
         LEV       0.92      0.90      0.91      9977
          MA       0.95      0.95      0.95      9977
          SA       0.85      0.86      0.86      9977

    accuracy                           0.90     49885
   macro avg       0.90      0.90      0.90     49885
weighted avg       0.90      0.90      0.90     49885



In [26]:
sample = ["شنو أحسن وقت أزور فيه تركيا؟"]
pred = model.predict(sample)
print("اللهجة المتوقعة:", le.inverse_transform(pred)[0])


اللهجة المتوقعة: SA


In [27]:
import joblib

# مسار الحفظ
model_path = r"C:\Users\Owner\Desktop\NLP\dialect_classifier_model.joblib"

# حفظ الـ Pipeline كامل
joblib.dump(model, model_path)

print("✅ تم حفظ النموذج بنجاح")


✅ تم حفظ النموذج بنجاح


In [35]:
text = ["مش عاوز"]  # ضع أي نص هنا

# التوقع
pred_id = model.predict(text)

# تحويل الرقم إلى اسم اللهجة
pred_label = le.inverse_transform(pred_id)

print("اللهجة المتوقعة:", pred_label[0])


اللهجة المتوقعة: EG
